# Feature engineering

In [64]:
import numpy as np
import pandas as pd
import os
import time

## Загрузка данных

In [65]:
train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

def remove_duplicates(df, name):
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    after = len(df)
    print(f"{name}: удалено {before - after} дубликатов")
    return df

train = remove_duplicates(train, "train")
test = remove_duplicates(test, "test")
events = remove_duplicates(events, "events")

print(f"  train:  {train.shape}")
print(f"  test:   {test.shape}")
print(f"  events: {events.shape}")

train: удалено 0 дубликатов
test: удалено 0 дубликатов
events: удалено 4863 дубликатов
  train:  (11091, 5)
  test:   (4909, 4)
  events: (324042, 14)


## Фильтруем события в пределах окна

In [66]:
def filter_window_events(events_df, meta_df):
    ev = events_df.merge(meta_df[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id', how='inner')
    ev = ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)].copy()
    return ev.drop(columns=['window_start_ts', 'window_end_ts'])

ev_tr = filter_window_events(events, train)
ev_te = filter_window_events(events, test)

print(f"  Событий в окне Train: {len(ev_tr)}")
print(f"  Событий в окне Test:  {len(ev_te)}")

  Событий в окне Train: 195484
  Событий в окне Test:  88349


In [67]:
# Список всех типов событий, для которых создаём счётчики и доли
ALL_EVENTS = [
    'item_view', 'search_results_view', 'photo_swipe',
    'contact_phone_show', 'contact_chat_open', 'contact_message_sent',
    'favorite_add', 'seller_page_view', 'login',
]

In [68]:
def prepare_event_df(ev_df: pd.DataFrame) -> pd.DataFrame:
    """Обогащает DataFrame событий техническими, временными
    и поведенческими маркерами, нужными для последующей агрегации.
    """
    ev = ev_df.copy()

    # Нормализация строк платформы
    ev['platform_clean'] = ev['platform'].astype(str).str.lower()
    ua_str = ev['user_agent'].astype(str).str.lower()

    # Технические маркеры из User-Agent
    ev['is_crawler_lib'] = ua_str.str.contains(
        'scrapy|curl|requests|urllib|http-client|fetch|python', regex=True
    ).astype(int)
    ev['is_headless'] = ua_str.str.contains(
        'headlesschrome|phantomjs|selenium|playwright', regex=True
    ).astype(int)
    ev['is_avito_app'] = ua_str.str.contains(
        'okhttp|avito/', regex=True
    ).astype(int)

    # Маркеры платформы
    ev['is_desktop_plat'] = ev['platform_clean'].isin(['desktop', 'web']).astype(int)
    ev['is_mobile_plat'] = ev['platform_clean'].isin(['android', 'ios', 'iphone']).astype(int)
    
    # Сортировка и временные дельты между текущим и предыдущим событием того же cookie
    ev = ev.sort_values(['cookie_id', 'event_ts'], kind='mergesort')
    ev['dt'] = (
        ev.groupby('cookie_id', sort=False)['event_ts']
        .diff()
        .dt.total_seconds()
    )
    # Порядковый номер события внутри cookie
    ev['pos'] = ev.groupby('cookie_id', sort=False).cumcount()

    # Курсор
    ev['has_pointer'] = ev['pointer_x'].notna().astype(int)

    # Предыдущее событие
    ev['prev_event_name'] = (
        ev.groupby('cookie_id', sort=False)['event_name'].shift(1)
    )

    # Ночные часы
    ev['is_night'] = ev['event_ts'].dt.hour.isin([1, 2, 3, 4, 5]).astype(int)

    # One-hot по типам событий
    for e in ALL_EVENTS:
        ev[f'ev_{e}'] = (ev['event_name'] == e).astype(int)

    # Тип продавца
    ev['is_private_seller'] = (ev['seller_type'] == 'private').astype(int)
    ev['is_pro_seller'] = (ev['seller_type'] == 'pro').astype(int)

    # Биграммы переходов
    ev['trans_item_to_photo'] = (
        (ev['prev_event_name'] == 'item_view') &
        (ev['event_name'] == 'photo_swipe')
    ).astype(int)
    ev['trans_item_to_phone'] = (
        (ev['prev_event_name'] == 'item_view') &
        (ev['event_name'] == 'contact_phone_show')
    ).astype(int)
    ev['trans_item_to_item'] = (
        (ev['prev_event_name'] == 'item_view') &
        (ev['event_name'] == 'item_view')
    ).astype(int)
    ev['trans_search_to_search'] = (
        (ev['prev_event_name'] == 'search_results_view') &
        (ev['event_name'] == 'search_results_view')
    ).astype(int)
    ev['trans_photo_to_photo'] = (
        (ev['prev_event_name'] == 'photo_swipe') &
        (ev['event_name'] == 'photo_swipe')
    ).astype(int)

    # Доля событий после первого логина
    first_login = (
        ev[ev['event_name'] == 'login']
        .groupby('cookie_id')['pos']
        .min()
    )
    ev['after_login'] = (
        ev['pos'] > ev['cookie_id'].map(first_login)
    ).astype(float)

    # Максимальная длина серии одинаковых событий подряд
    ev['_chg'] = ev['event_name'].ne(
        ev.groupby('cookie_id', sort=False)['event_name'].shift()
    )
    ev['_run'] = ev.groupby('cookie_id', sort=False)['_chg'].cumsum()

    return ev

In [69]:
def agg_base(ev: pd.DataFrame, g: pd.core.groupby.DataFrameGroupBy) -> pd.DataFrame:
    """Считает основные агрегаты по cookie_id:
    - число событий, длительность сессии, частоту;
    - статистики по dt (mean, std, median, квантили, доли быстрых/медленных);
    - ночную активность;
    - число уникальных часов активности.
    """
    f = pd.DataFrame(index=g.indices.keys())
    f.index.name = 'cookie_id'

    # Общие счётчики
    f['n_events'] = g.size()
    f['active_duration_sec'] = (
        (g['event_ts'].max() - g['event_ts'].min()).dt.total_seconds()
    )
    f['events_per_sec'] = f['n_events'] / (f['active_duration_sec'] + 1.0)

    # Статистики по dt
    f['dt_mean'] = g['dt'].mean()
    f['dt_std'] = g['dt'].std().fillna(0)
    f['dt_median'] = g['dt'].median()
    f['dt_min'] = g['dt'].min()
    f['dt_q25'] = g['dt'].quantile(0.25)
    f['dt_q75'] = g['dt'].quantile(0.75)
    f['dt_iqr'] = f['dt_q75'] - f['dt_q25']
    # Доля переходов быстрее 30 с
    f['dt_fast_ratio_30s'] = g['dt'].apply(lambda x: (x < 30.0).mean())
    # Доля переходов быстрее 1 с
    f['dt_fast_ratio_1s'] = g['dt'].apply(lambda x: (x < 1.0).mean())
    # Доля переходов быстрее 0.3 с
    f['dt_ultrafast_ratio_03s'] = g['dt'].apply(lambda x: (x < 0.3).mean())

    # Доля dt, попадающих точно в +-0.05 с от 1 с
    f['dt_int_bias_1s'] = g['dt'].apply(
        lambda x: (np.abs(x - 1.0) < 0.05).mean()
    )

    # Ночная активность
    f['night_events_cnt'] = g['is_night'].sum()
    f['night_events_ratio'] = f['night_events_cnt'] / f['n_events']
    f['active_hours_nunique'] = g['event_ts'].apply(lambda s: s.dt.hour.nunique())

    return f

In [70]:
def agg_pointer_and_search(
    ev: pd.DataFrame,
    g: pd.core.groupby.DataFrameGroupBy,
    f: pd.DataFrame,
) -> pd.DataFrame:
    """Признаки указателя мыши (pointer) и глубины пагинации каталога."""

    # Курсор
    f['pointer_ratio'] = g['has_pointer'].mean()
    f['pointer_x_std'] = g['pointer_x'].std().fillna(0)
    f['pointer_y_std'] = g['pointer_y'].std().fillna(0)
    # Число уникальных пар (x, y)
    f['pointer_unique_pts'] = g.apply(
        lambda d: len(set(zip(d['pointer_x'].dropna(), d['pointer_y'].dropna())))
    )

    # Пагинация каталога
    f['search_page_max'] = g['search_page'].max().fillna(0)
    f['search_page_mean'] = g['search_page'].mean().fillna(0)
    f['search_page_std'] = g['search_page'].std().fillna(0)
    f['search_page_nunique'] = g['search_page'].nunique()
    f['is_deep_search_5'] = (f['search_page_max'] > 5).astype(int)
    f['is_deep_search_10'] = (f['search_page_max'] > 10).astype(int)
    f['is_deep_search_20'] = (f['search_page_max'] >= 20).astype(int)

    return f

In [71]:
def agg_entities_and_ua(
    g: pd.core.groupby.DataFrameGroupBy,
    f: pd.DataFrame,
) -> pd.DataFrame:
    """Уникальные объявления, категории, локации, поисковые запросы,
    типы продавцов, характеристики User-Agent.
    """

    # Уникальные сущности
    f['item_nunique'] = g['item_id'].nunique()
    f['category_nunique'] = g['item_category'].nunique()
    f['location_nunique'] = g['item_location'].nunique()
    f['is_multi_location'] = (f['location_nunique'] > 5).astype(int)

    # Поисковые запросы
    f['query_nunique'] = g['search_query'].nunique()
    f['items_per_query'] = f['item_nunique'] / (f['query_nunique'] + 1.0)

    # Продавцы
    f['seller_private_cnt'] = g['is_private_seller'].sum()
    f['seller_pro_cnt'] = g['is_pro_seller'].sum()
    f['seller_private_ratio'] = f['seller_private_cnt'] / (f['item_nunique'] + 1e-5)
    f['seller_pro_ratio'] = f['seller_pro_cnt'] / (f['item_nunique'] + 1e-5)

    # User-Agent / платформа
    f['is_crawler_lib_max'] = g['is_crawler_lib'].max()
    f['is_headless_max'] = g['is_headless'].max()
    f['is_avito_app_max'] = g['is_avito_app'].max()
    f['is_desktop_max'] = g['is_desktop_plat'].max()
    # Десктоп без единого события с указателем
    f['desktop_no_pointer'] = (
        (f['is_desktop_max'] == 1) & (f['pointer_ratio'] == 0)
    ).astype(int)
    f['ua_nunique'] = g['user_agent'].nunique()

    return f

In [72]:
def agg_event_counts_and_funnel(
    g: pd.core.groupby.DataFrameGroupBy,
    f: pd.DataFrame,
) -> pd.DataFrame:
    """Абсолютные счётчики и доли каждого типа событий.
    Счётчики биграмм переходов.
    """

    # Счётчики и доли
    for e in ALL_EVENTS:
        f[f'cnt_{e}'] = g[f'ev_{e}'].sum()
        f[f'ratio_{e}'] = f[f'cnt_{e}'] / f['n_events']

    # Воронка
    f['photo_to_item_ratio'] = f['cnt_photo_swipe'] / (f['cnt_item_view'] + 1.0)
    f['phone_to_item_ratio'] = f['cnt_contact_phone_show'] / (f['cnt_item_view'] + 1.0)
    f['chat_to_item_ratio'] = f['cnt_contact_chat_open'] / (f['cnt_item_view'] + 1.0)
    # Доля повторных просмотров одного и того же объявления
    f['item_repeat_ratio'] = 1.0 - (f['item_nunique'] / (f['cnt_item_view'] + 1e-5))
    f['total_contacts'] = (
        f['cnt_contact_phone_show'] +
        f['cnt_contact_chat_open'] +
        f['cnt_contact_message_sent']
    )
    f['total_contacts_ratio'] = f['total_contacts'] / f['n_events']

    # Биграммы
    f['trans_item_to_photo_cnt'] = g['trans_item_to_photo'].sum()
    f['trans_item_to_photo_ratio'] = (
        f['trans_item_to_photo_cnt'] / (f['cnt_item_view'] + 1.0)
    )
    f['trans_item_to_phone_cnt'] = g['trans_item_to_phone'].sum()
    f['trans_item_to_phone_ratio'] = (
        f['trans_item_to_phone_cnt'] / (f['cnt_item_view'] + 1.0)
    )
    f['trans_item_to_item_cnt'] = g['trans_item_to_item'].sum()
    f['trans_search_to_search_cnt'] = g['trans_search_to_search'].sum()
    f['trans_photo_to_photo_cnt'] = g['trans_photo_to_photo'].sum()

    return f

In [73]:
def agg_catalog_traversal(
    ev: pd.DataFrame,
    g: pd.core.groupby.DataFrameGroupBy,
    f: pd.DataFrame,
) -> pd.DataFrame:
    """Признаки, специфичные для обхода каталога:
    - page_arith_run: максимальная длина серии страниц с шагом +1
    - items_per_search: сколько уникальных объявлений приходится
      на одно событие search_results_view;
    - item_pop_mean / item_pop_min: средняя и минимальная популярность
      просмотренных объявлений
    """

    # Арифметическая серия страниц
    search_df = (
        ev.dropna(subset=['search_page'])[['cookie_id', 'search_page']]
        .copy()
        .sort_values(['cookie_id', 'search_page'], kind='mergesort')
    )
    if len(search_df) > 0:
        # inc = True, если страница ровно на 1 больше предыдущей
        search_df['inc'] = (
            search_df.groupby('cookie_id', sort=False)['search_page']
            .diff()
            .eq(1)
        )
        # Разбиваем на runs подряд идущих True / False
        search_df['_chg'] = search_df['inc'].ne(
            search_df.groupby('cookie_id', sort=False)['inc'].shift()
        )
        search_df['_run'] = (
            search_df.groupby('cookie_id', sort=False)['_chg'].cumsum()
        )
        run_len = search_df.groupby(
            ['cookie_id', '_run', 'inc'], sort=False
        ).size()
        if True in run_len.index.get_level_values('inc'):
            f['page_arith_run'] = (
                run_len.xs(True, level='inc', drop_level=False)
                .groupby(level=0)
                .max()
            )
        else:
            f['page_arith_run'] = 0.0
    else:
        f['page_arith_run'] = 0.0
    f['page_arith_run'] = f['page_arith_run'].fillna(0.0)

    # Items per search
    f['items_per_search'] = f['item_nunique'] / (f['cnt_search_results_view'] + 1.0)
    
    # Популярность объявлений
    item_pop_df = ev.dropna(subset=['item_id']).copy()
    
    if len(item_pop_df) > 0:
        # Маппим каждому item_id число уникальных cookie, которые его смотрели
        local_pop = item_pop_df.groupby('item_id')['cookie_id'].nunique()
        item_pop_df['_pop'] = item_pop_df['item_id'].map(local_pop).fillna(0.0)
        pop_agg = item_pop_df.groupby('cookie_id', sort=False).agg(
            item_pop_mean=('_pop', 'mean'),
            item_pop_min=('_pop', 'min'),
        )
        f['item_pop_mean'] = np.log1p(pop_agg['item_pop_mean'].fillna(0.0))
        f['item_pop_min'] = np.log1p(pop_agg['item_pop_min'].fillna(0.0))
    else:
        f['item_pop_mean'] = 0.0
        f['item_pop_min'] = 0.0
    f['item_pop_mean'] = f['item_pop_mean'].fillna(0.0)
    f['item_pop_min'] = f['item_pop_min'].fillna(0.0)

    return f

In [74]:
def agg_behaviour(
    ev: pd.DataFrame,
    g: pd.core.groupby.DataFrameGroupBy,
    f: pd.DataFrame,
) -> pd.DataFrame:
    """Дополнительные поведенческие метрики"""

    f['fav_per_search'] = f['cnt_favorite_add'] / (f['cnt_search_results_view'] + 1.0)
    f['contact_per_search'] = f['total_contacts'] / (f['cnt_search_results_view'] + 1.0)
    f['events_after_login_share'] = g['after_login'].mean().fillna(0.0)

    # Максимальная длина серии
    run_sizes = ev.groupby(['cookie_id', '_run'], sort=False).size()
    run_len_max_series = run_sizes.groupby('cookie_id', sort=False).max()
    f['run_len_max'] = run_len_max_series.reindex(f.index, fill_value=0.0)

    # Энтропия по долям типов событий
    # Собираем вектор долей; clip, чтобы log не ушёл в -inf
    shares = np.column_stack(
        [f[f'ratio_{e}'].fillna(0.0).values for e in ALL_EVENTS]
    )
    shares_clip = np.clip(shares, 1e-12, 1.0)
    f['event_entropy'] = -(shares_clip * np.log(shares_clip)).sum(axis=1)

    return f

In [75]:
def agg_pagination(
    ev: pd.DataFrame,
    f: pd.DataFrame,
) -> pd.DataFrame:
    """Дополнительные признаки пагинации:
    - page_jump_mean — средний модуль скачка между страницами;
    - page_skip_share — доля скачков > 1 (пропуск страниц);
    - no_text_search — флаг полного отсутствия поисковых событий.
    """

    search_ev = ev.dropna(subset=['search_page'])

    if len(search_ev) > 0:
        page_jump = (
            search_ev
            .groupby('cookie_id', sort=False)['search_page']
            .diff()
            .abs()
        )
        f['page_jump_mean'] = page_jump.groupby(
            search_ev['cookie_id'], sort=False
        ).mean()
        f['page_skip_share'] = page_jump.gt(1).groupby(
            search_ev['cookie_id'], sort=False
        ).mean()
    else:
        f['page_jump_mean'] = 0.0
        f['page_skip_share'] = 0.0

    f['page_jump_mean'] = f['page_jump_mean'].fillna(0.0)
    f['page_skip_share'] = f['page_skip_share'].fillna(0.0)

    f['no_text_search'] = (f['cnt_search_results_view'] == 0).astype(int)

    return f

In [76]:
def merge_meta(
    f: pd.DataFrame,
    meta_df: pd.DataFrame,
) -> pd.DataFrame:
    """Слияние агрегатов с метаданными cookie и добавление:
    - возраста cookie (часы, дни, log);
    - флагов свежей cookie и ночного создания;
    - кросс-признаков;
    - рангов внутри одного дня наблюдения.
    """

    res = meta_df[
        ['cookie_id', 'cookie_created_at', 'window_start_ts', 'window_end_ts']
    ].merge(f.reset_index(), on='cookie_id', how='left')

    # Возраст cookie
    res['cookie_age_hours'] = (
        (res['window_start_ts'] - res['cookie_created_at'])
        .dt.total_seconds() / 3600.0
    )
    res['cookie_age_days'] = res['cookie_age_hours'] / 24.0
    res['log_cookie_age_hours'] = np.log1p(np.maximum(0, res['cookie_age_hours']))
    res['is_fresh_cookie_24h'] = (res['cookie_age_hours'] < 24.0).astype(int)
    res['is_fresh_cookie_1h'] = (res['cookie_age_hours'] < 1.0).astype(int)

    # Час создания cookie
    res['created_hour'] = res['cookie_created_at'].dt.hour
    res['is_created_at_night'] = res['created_hour'].isin([0, 1, 2, 3, 4, 5]).astype(int)
    res['is_created_cron_peak'] = res['created_hour'].isin([10, 15, 19, 21]).astype(int)

    #  Кросс-признаки
    res['item_uniq_x_new'] = (
        res['item_repeat_ratio'] *
        (res['cookie_age_days'] < 7.0).astype(float)
    ).fillna(0.0)

    # Много страниц и локаций, нет логина,
    # нормализованные на возраст cookie (насколько интенсивно обходились страницы)
    res['traverse_intensity'] = (
        (1.0 - (res['cnt_login'] > 0).astype(float))
        * np.log1p(res['search_page_max'])
        * np.log1p(res['location_nunique'])
        / (res['cookie_age_days'].clip(lower=0) + 1.0)
    ).fillna(0.0)

    # Логин × другие
    res['has_login'] = (res['cnt_login'] > 0).astype(int)
    res['login_x_locations'] = res['has_login'] * res['location_nunique']
    res['login_x_events'] = res['has_login'] * res['n_events']

    # Ранжирование внутри дня наблюдения
    # если сегодня все cookie просматривали < 5 страниц, а эта — 50,
    # ранг будет близок к 1.0.
    res['_day'] = res['window_start_ts'].dt.date

    peer_cols = [
        'n_events', 'location_nunique', 'item_nunique',
        'search_page_max', 'events_per_sec', 'dt_median',
    ]
    if 'item_pop_mean' in res.columns:
        peer_cols.append('item_pop_mean')

    for col in peer_cols:
        if col in res.columns:
            res[f'peer_{col}'] = res.groupby('_day')[col].rank(pct=True)

    res.drop(columns=['_day'], inplace=True)

    return res

In [77]:
def extract_cookie_features(
    ev_df: pd.DataFrame,
    meta_df: pd.DataFrame,
    is_train: bool = True,
) -> pd.DataFrame:
    """Строит матрицу признаков по cookie_id."""

    ev = prepare_event_df(ev_df)
    g = ev.groupby('cookie_id', sort=False)
    f = agg_base(ev, g)
    f = agg_pointer_and_search(ev, g, f)
    f = agg_entities_and_ua(g, f)
    f = agg_event_counts_and_funnel(g, f)
    f = agg_catalog_traversal(ev, g, f)
    f = agg_behaviour(ev, g, f)
    f = agg_pagination(ev, f)
    res = merge_meta(f, meta_df)

    # Добавляем target (только для обучения) 
    if is_train and 'target' in meta_df.columns:
        res['target'] = meta_df['target'].values

    return res

In [78]:
df_train_features = extract_cookie_features(ev_tr, train, is_train=True)
df_test_features = extract_cookie_features(ev_te, test, is_train=False)

print("\nРазмерности полученных матриц признаков:")
print(f"  df_train_features: {df_train_features.shape}")
print(f"  df_test_features:  {df_test_features.shape}")


Размерности полученных матриц признаков:
  df_train_features: (11091, 112)
  df_test_features:  (4909, 111)


Сохраняем

In [79]:
os.makedirs('features', exist_ok=True)

# Сохраняем в Parquet и дублируем в CSV
df_train_features.to_parquet('features/train_features.parquet', index=False)
df_test_features.to_parquet('features/test_features.parquet', index=False)

df_train_features.to_csv('features/train_features.csv', index=False)
df_test_features.to_csv('features/test_features.csv', index=False)

В итоге получили около 100 новых признаков, которые в дальнейшем пойдут на обучение, при этом пропуски не обрабатывались потому что все зависит от выбраного алгоритма